# Sentiment Analysis and Interpretation on Tweets Using BERT and Captum

In this notebook, we'll create an application for sentiment analysis on tweets using a BERT model.
We'll use Captum's Integrated Gradients to interpret the results, providing insight into which words in each tweet contribute most to the sentiment prediction.

## Setup

First, we install the necessary libraries: Hugging Face Transformers for the BERT model and Captum for interpretability. If you’re running this locally or in Colab, make sure to run the following cell to install these packages.

In [ ]:
!pip install transformers captum

## Import Libraries

Now we import the essential libraries, including `BertForSequenceClassification` for loading a pre-trained BERT model, `BertTokenizer` for tokenizing text, `IntegratedGradients` from Captum for attribution, and additional libraries for data handling and visualization.

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from captum.attr import IntegratedGradients
import matplotlib.pyplot as plt
import numpy as np

## Load Model and Tokenizer

We use a pretrained BERT model fine-tuned for sentiment analysis, specifically the `nlptown/bert-base-multilingual-uncased-sentiment` model. This model classifies sentiment on a scale from 1 (very negative) to 5 (very positive).

In [ ]:
model = BertForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
tokenizer = BertTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

## Define a Function for Sentiment Analysis with Interpretation

We define a function that takes a tweet, analyzes its sentiment, and computes word attributions using Captum's Integrated Gradients. The function returns the sentiment score and a plot of word attributions for each tweet.

In [ ]:
def analyze_tweet_sentiment(tweet, target_sentiment=4):
    # Tokenize and get input embeddings
    inputs = tokenizer(tweet, return_tensors='pt', truncation=True, padding=True)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    # Convert input_ids to embeddings
    embeddings = model.bert.embeddings(input_ids)
    embeddings.requires_grad_(True)
    baseline = torch.zeros_like(embeddings)

    # Initialize Integrated Gradients
    ig = IntegratedGradients(lambda x: model(inputs_embeds=x, attention_mask=attention_mask).logits)

    # Compute attributions
    attributions = ig.attribute(inputs=embeddings, baselines=baseline, target=target_sentiment)
    attributions = attributions.sum(dim=2).squeeze(0).detach().numpy()

    # Decode tokens and merge sub-token attributions
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())

    merged_tokens = []
    merged_attributions = []
    current_word = ''
    current_attribution = 0.0

    for token, attribution in zip(tokens, attributions):
        if token == '[CLS]' or token == '[SEP]':
            continue  # Skip special tokens
        elif token.startswith('##'):
            current_word += token[2:]  # Remove '##' and add to the current word
            current_attribution += attribution
        else:
            if current_word:
                merged_tokens.append(current_word)
                merged_attributions.append(current_attribution)
            current_word = token
            current_attribution = attribution

    if current_word:
        merged_tokens.append(current_word)
        merged_attributions.append(current_attribution)

    # Normalize attributions for visualization
    merged_attributions = (np.array(merged_attributions) - np.min(merged_attributions)) / (
        np.max(merged_attributions) - np.min(merged_attributions) + 1e-5
    )

    # Plot attributions
    plt.figure(figsize=(10, 2))
    plt.bar(range(len(merged_tokens)), merged_attributions, color='blue', alpha=0.7)
    plt.xticks(range(len(merged_tokens)), merged_tokens, rotation=90)
    plt.xlabel('Words')
    plt.ylabel('Attribution')
    plt.title('Word Attributions for Sentiment Prediction')
    plt.show()

    # Get the predicted sentiment label
    output = model(inputs['input_ids'], attention_mask=attention_mask).logits
    predicted_sentiment = torch.argmax(output, dim=1).item() + 1  # Adding 1 to match 1-5 sentiment scale

    print(f'Predicted Sentiment: {predicted_sentiment}/5')

## Analyze Sample Tweets

Now, we’ll test the function with a few sample tweets. You can modify the tweet text or add your own to see different results.

In [ ]:
tweets = [
    "I absolutely love this product, it's fantastic!",
    "This is a terrible experience, I am very disappointed.",
    "I'm neutral about this, neither good nor bad.",
    "Captum makes it incredibly easy to understand complex models!",
    "The new update is okay, but could be improved."
]

# Analyze each tweet
for tweet in tweets:
    print(f'\nAnalyzing tweet: {tweet}')
    analyze_tweet_sentiment(tweet)

## Interpretation

For each tweet, the model predicts a sentiment score on a scale of 1-5, where:
- **1** = Very Negative
- **2** = Negative
- **3** = Neutral
- **4** = Positive
- **5** = Very Positive

The word attributions indicate which words contribute most to the model’s prediction. Words with high attribution scores strongly influence the sentiment rating, while words with low attribution scores have minimal impact on the prediction.

For example:
- **Positive sentiment words** like "love" or "great" are expected to have high attribution values in positive tweets.
- **Negative sentiment words** like "hate" or "terrible" are expected to have high attribution values in negative tweets.
- **Neutral words** such as "the" or "and" or "okay" typically have lower attributions since they contribute little to the overall sentiment.

This analysis helps us understand not only the predicted sentiment score but also which specific words or phrases led the model to that conclusion.



### Example Predictions:

1. **Tweet**: "I absolutely love this product, it's fantastic!"
   - **Predicted Sentiment**: 5/5 (Very Positive)
   - **Explanation**: Words like **"love"** and **"fantastic"** carry strong positive sentiment, which likely contributed to the high score. This prediction aligns well with the tweet’s tone.

2. **Tweet**: "This is a terrible experience, I am very disappointed."
   - **Predicted Sentiment**: 1/5 (Very Negative)
   - **Explanation**: The words **"terrible"** and **"disappointed"** are strongly associated with negative sentiment. The model correctly identifies this tweet as very negative.

3. **Tweet**: "I'm neutral about this, neither good nor bad."
   - **Predicted Sentiment**: 1/5 (Very Negative)
   - **Explanation**: Despite its neutral tone, this tweet is misclassified as very negative. This misclassification likely occurs because the model associates words like **"neither"** and **"nor"** with negative sentiment, perhaps due to patterns in its training data. Additionally, BERT-based models may struggle with nuanced or balanced language, as they rely heavily on patterns observed during training, which may not have had enough examples of explicitly neutral phrases.

4. **Tweet**: "Captum makes it incredibly easy to understand complex models!"
   - **Predicted Sentiment**: 5/5 (Very Positive)
   - **Explanation**: Positive words like **"incredibly easy"** and **"understand"** contribute to the positive sentiment rating. The model accurately interprets this as a very positive tweet.

5. **Tweet**: "The new update is okay, but could be improved."
   - **Predicted Sentiment**: 3/5 (Neutral)
   - **Explanation**: The word **"okay"** carries a mild positive connotation, while **"could be improved"** suggests some dissatisfaction. Together, these create a balanced tone, and the model correctly assigns a neutral score.

### Addressing Misclassification of Neutral Sentiment

To improve the model's ability to accurately classify neutral statements, consider:
1. **Fine-Tuning on More Neutral Data**: Incorporating more examples of neutral or balanced statements in the training data can help the model recognize phrases with neutral sentiment.
2. **Data Augmentation**: Adding variations of neutral phrases like "indifferent" or "no opinion" may help reinforce the concept of neutrality.
3. **Custom Rule-Based Post-Processing**: For applications where neutral accuracy is critical, applying a rule-based approach to adjust predictions on phrases like "neither good nor bad" or "I'm neutral" may improve performance.

This analysis not only highlights the model's predicted sentiment but also provides insight into which specific words or phrases influenced each prediction.
